# Step 7 — Machine Learning Models Training & Evaluation
**Project**: Deep Learning-Based Flood Prediction Using Rainfall Data  
**Dataset**: Feature-Engineered Dataset (`data/processed/`) (8,000 train / 2,000 test samples)  
**Objective**: Train, evaluate, and benchmark four baseline classification models (Logistic Regression, Decision Tree, Random Forest, XGBoost) to establish empirical performance standards prior to Deep Learning / LSTM development.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root directory to path
sys.path.append('..')
from src.train_ml import (
    load_processed_data,
    train_logistic_regression,
    train_decision_tree,
    train_random_forest,
    train_xgboost,
    evaluate_model,
    plot_and_save_confusion_matrix,
    plot_and_save_feature_importance,
    plot_and_save_model_comparison,
    train_and_evaluate_all_models
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Target Class Imbalance Audit
Verify class balance across training and test target vectors prior to model fitting.

In [2]:
X_train, X_test, y_train, y_test, feature_names = load_processed_data('../data/processed')

print('--- Target Class Balance Audit ---')
train_counts = y_train.value_counts()
test_counts = y_test.value_counts()
print(f"y_train Distribution : Flood (1): {train_counts[1]} ({train_counts[1]/len(y_train)*100:.2f}%), No Flood (0): {train_counts[0]} ({train_counts[0]/len(y_train)*100:.2f}%)")
print(f"y_test Distribution  : Flood (1): {test_counts[1]} ({test_counts[1]/len(y_test)*100:.2f}%), No Flood (0): {test_counts[0]} ({test_counts[0]/len(y_test)*100:.2f}%)")
print("STATUS: Target distribution is perfectly balanced (~50/50). Artificial resampling (SMOTE) is unnecessary.")

--- Target Class Balance Audit ---
y_train Distribution : Flood (1): 4046 (50.58%), No Flood (0): 3954 (49.43%)
y_test Distribution  : Flood (1): 1011 (50.55%), No Flood (0): 989 (49.45%)
STATUS: Target distribution is perfectly balanced (~50/50). Artificial resampling (SMOTE) is unnecessary.


## 2. Model 1 — Logistic Regression
Linear baseline classifier trained on scaled features.

In [3]:
model_lr = train_logistic_regression(X_train, y_train)
metrics_lr, cm_lr = evaluate_model(model_lr, X_test, y_test, 'Logistic Regression')
print(metrics_lr)
plot_and_save_confusion_matrix(cm_lr, 'Logistic Regression', '../results')

{'Model': 'Logistic Regression', 'Accuracy': 0.5075, 'Precision': 0.5116, 'Recall': 0.5668, 'F1-score': 0.5378}


## 3. Model 2 — Decision Tree Classifier
Non-linear single-tree decision rule classifier (`max_depth=10`).

In [4]:
model_dt = train_decision_tree(X_train, y_train, max_depth=10)
metrics_dt, cm_dt = evaluate_model(model_dt, X_test, y_test, 'Decision Tree')
print(metrics_dt)
plot_and_save_confusion_matrix(cm_dt, 'Decision Tree', '../results')

{'Model': 'Decision Tree', 'Accuracy': 0.504, 'Precision': 0.5093, 'Recall': 0.5153, 'F1-score': 0.5123}


## 4. Model 3 — Random Forest Classifier
Ensemble bagging decision trees (`n_estimators=100`, `max_depth=15`).

In [5]:
model_rf = train_random_forest(X_train, y_train, n_estimators=100, max_depth=15)
metrics_rf, cm_rf = evaluate_model(model_rf, X_test, y_test, 'Random Forest')
print(metrics_rf)
plot_and_save_confusion_matrix(cm_rf, 'Random Forest', '../results')

# Feature Importance
imp_rf = pd.Series(model_rf.feature_importances_, index=feature_names)
plot_and_save_feature_importance(imp_rf, 'Random Forest', top_n=15, output_dir='../results')

{'Model': 'Random Forest', 'Accuracy': 0.505, 'Precision': 0.5098, 'Recall': 0.5381, 'F1-score': 0.5236}


## 5. Model 4 — XGBoost Classifier
Gradient boosted decision trees (`n_estimators=100`, `learning_rate=0.1`, `max_depth=6`).

In [6]:
model_xgb = train_xgboost(X_train, y_train, n_estimators=100, learning_rate=0.1, max_depth=6)
metrics_xgb, cm_xgb = evaluate_model(model_xgb, X_test, y_test, 'XGBoost')
print(metrics_xgb)
plot_and_save_confusion_matrix(cm_xgb, 'XGBoost', '../results')

# Feature Importance
imp_xgb = pd.Series(model_xgb.feature_importances_, index=feature_names)
plot_and_save_feature_importance(imp_xgb, 'XGBoost', top_n=15, output_dir='../results')

{'Model': 'XGBoost', 'Accuracy': 0.517, 'Precision': 0.521, 'Recall': 0.5519, 'F1-score': 0.536}


## 6. Model Performance Comparison & Artifact Export

In [7]:
metrics_df, models_dict, cms_dict = train_and_evaluate_all_models(
    data_dir='../data/processed',
    output_dir='../results',
    models_dir='../models'
)

print('='*65)
print('MACHINE LEARNING MODEL COMPARISON TABLE')
print('='*65)
print(metrics_df.to_string(index=False))
print('='*65)

MACHINE LEARNING MODEL COMPARISON TABLE
              Model  Accuracy  Precision  Recall  F1-score
Logistic Regression    0.5075     0.5116  0.5668    0.5378
      Decision Tree    0.5040     0.5093  0.5153    0.5123
      Random Forest    0.5050     0.5098  0.5381    0.5236
            XGBoost    0.5170     0.5210  0.5519    0.5360


## 7. Step 7 Final Summary & Benchmark Analysis
1. **Best Model by F1-Score**: Logistic Regression (0.5378) / XGBoost (0.5360)
2. **Best Model by Recall**: Logistic Regression (0.5668) / XGBoost (0.5519)
3. **Best Model by Accuracy**: XGBoost (0.5170)
4. **Class Imbalance Impact**: None. Target class is perfectly balanced (~50.5% / ~49.5%).
5. **Baseline Model Recommendation**: **XGBoost (Accuracy 0.5170, F1 0.5360)** and **Logistic Regression (Recall 0.5668)** serve as the official ML baseline benchmarks for the subsequent Deep Learning (LSTM) stage.